In [1]:
import tensorflow as tf
import numpy as np

diffusion_steps = 16


2025-04-26 18:08:47.017837: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-26 18:08:47.051840: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-26 18:08:47.051867: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-26 18:08:47.052768: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-26 18:08:47.058415: I tensorflow/core/platform/cpu_feature_guar

In [2]:
t = tf.cast(tf.range(diffusion_steps + 1), tf.float32)
alpha_bar = tf.cos(0.5 * np.pi * t / diffusion_steps) ** 2

2025-04-26 18:08:50.630823: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 937 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:19:00.0, compute capability: 8.6
2025-04-26 18:08:50.631384: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 1755 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:1a:00.0, compute capability: 8.6
2025-04-26 18:08:50.631913: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 8244 MB memory:  -> device: 2, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:68:00.0, compute capability: 8.6
2025-04-26 18:08:50.799518: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


In [5]:
import soundfile as sf

filepath_a = "data/out_combined/anechoic/Actions - Devil's Words_chunk0_aug0.wav"
filepath_r = "data/out_combined/reverb/Actions - Devil's Words_chunk0_aug0.wav"

anechoic, _ = sf.read(filepath_a)
reverb, _ = sf.read(filepath_r)


In [10]:
def compute_ri(signal):
        """Compute RI STFTs for UNet RI, DCCRN, DCUNet."""
        signal_stft = tf.signal.stft(
            signal,
            frame_length=510, #510
            frame_step=177, #177
            fft_length=510, #510
            window_fn=tf.signal.hann_window,
        )

        signal_stft_real = tf.math.real(signal_stft)
        signal_stft_imag = tf.math.imag(signal_stft)

        # create a new dimension also for UNet
        signal_stft_ri = tf.stack([signal_stft_real, signal_stft_imag], axis=-1)

        return tf.cast(signal_stft_ri, tf.float32)


def diffusion(reverb, clean, timestep):

        # diffed_mag = timestep * clean_mag + (1 - tf.sqrt(timestep)) * reverb_mag
        diffed_mag = timestep * clean + (1 - timestep) * reverb
        return diffed_mag


def compute_signal_from_RI_stft(ri_stft):
    
    polar_spec = tf.complex(ri_stft[...,0], ri_stft[...,1])
        
    inversed_stft = tf.signal.inverse_stft(polar_spec, frame_length=510, frame_step=177, 
                fft_length=510, window_fn=tf.signal.inverse_stft_window_fn(
                     177,forward_window_fn=tf.signal.hann_window))
    return inversed_stft


In [11]:
reverb_ri_stft = compute_ri(reverb)
clean_ri_stft = compute_ri(anechoic)

out_path = './examples/'

In [12]:
for i in range(0, diffusion_steps+1):
    
    timestep = tf.constant(i, dtype=tf.int32, shape=[1])

    #apply that to get the corresponding alpha value
    noise_level = tf.cast(tf.gather(alpha_bar, timestep), tf.float32)

    #apply cold diffusion on RI
    noised = diffusion(reverb_ri_stft, clean_ri_stft, noise_level)
    
    #invert
    noised_wav = compute_signal_from_RI_stft(noised)
    
    sf.write(out_path+'diffused_ri_'+str(i)+'.wav', noised_wav.numpy(), 44100)

In [19]:
def good_for_unet(signal_length, n_fft, hop, depth):
    # how many frames your STFT will produce
    T = (signal_length - n_fft) // hop + 1
    return T % (2**depth) == 0

# example:
signal_length = 88200  # or however many samples you're feeding in
n_fft = 1022
for hop in (341, 256, 128, 128):
    print(hop, (signal_length - n_fft)//hop + 1, "→",
          "OK" if good_for_unet(signal_length,n_fft,hop,depth=4) else "FAIL")

341 256 → OK
256 341 → FAIL
128 682 → FAIL
128 682 → FAIL


In [ ]:
def compute_ri(signal):
        """Compute RI STFTs for UNet RI, DCCRN, DCUNet."""
        signal_stft = tf.signal.stft(
            signal,
            frame_length=win,
            frame_step=hop,
            fft_length=fft,
            window_fn=window_fn(),
        )

        signal_stft_real = tf.math.real(signal_stft)
        signal_stft_imag = tf.math.imag(signal_stft)

        # create a new dimension also for UNet
        signal_stft_ri = tf.stack([signal_stft_real, signal_stft_imag], axis=-1)

        return tf.cast(signal_stft_ri, tf.float32)